## Example 1: Creating images and basic transforms

In [ ]:
from PIL import Image

# Example 1: Basic image creation
print("Example 1: Basic image creation")

print(f"Pillow version: {Image.__version__}")

# Create images with different modes
modes = [
    ('RGB', (255, 0, 0)),
    ('RGBA', (0, 255, 0, 128)),
    ('L', 128),
]

expected_bands = {'RGB': ('R', 'G', 'B'), 'RGBA': ('R', 'G', 'B', 'A'), 'L': ('L',)}
for mode, color in modes:
    img = Image.new(mode, (100, 100), color=color)
    print(f"Created {mode}: {img.size}, bands: {img.getbands()}")
    assert img.size == (100, 100) and img.mode == mode
    assert img.getbands() == expected_bands[mode], img.getbands()
    assert img.getpixel((50, 50)) == color, img.getpixel((50, 50))

# Basic operations
img = Image.new('RGB', (400, 300), color='lightblue')
resized = img.resize((200, 150))
cropped = img.crop((50, 50, 250, 200))
rotated = img.rotate(45)

print(f"Resized: {resized.size}, Cropped: {cropped.size}, Rotated: {rotated.size}")
assert resized.size == (200, 150), resized.size
assert cropped.size == (200, 150), cropped.size
assert rotated.size == (400, 300), rotated.size  # rotate() keeps the size unless expand=True
assert img.rotate(45, expand=True).size[0] > 400
assert img.getpixel((0, 0)) == (173, 216, 230)  # 'lightblue'

print("Example 1 completed")

## Example 2: Drawing shapes and compositing

In [ ]:
from PIL import Image, ImageDraw

# Example 2: Drawing shapes
print("Example 2: Drawing shapes")

canvas = Image.new('RGB', (400, 300), color='white')
draw = ImageDraw.Draw(canvas)

# Draw shapes
draw.rectangle([50, 50, 150, 100], fill='red', outline='darkred', width=2)
draw.ellipse([200, 50, 300, 150], fill='blue', outline='darkblue', width=2)
draw.line([50, 150, 350, 200], fill='green', width=3)
draw.polygon([(100, 200), (150, 250), (50, 250)], fill='yellow', outline='orange')
draw.arc([250, 180, 350, 280], start=0, end=180, fill='purple', width=3)
draw.text((20, 20), "Pillow Demo", fill='black')

print(f"Drew shapes on {canvas.size} canvas")
assert canvas.getpixel((100, 75)) == (255, 0, 0)   # inside the rectangle
assert canvas.getpixel((250, 100)) == (0, 0, 255)  # inside the ellipse
assert canvas.getpixel((390, 10)) == (255, 255, 255)  # background

# Create composite with transparent layers
composite = Image.new('RGBA', (200, 200), (255, 255, 255, 255))
layer1 = Image.new('RGBA', (150, 150), (255, 0, 0, 128))
composite.paste(layer1, (25, 25), layer1)

print("Created composite with transparency")
# Half-transparent red over white gives pink inside the layer; outside it stays white.
r, g, b, a = composite.getpixel((100, 100))
assert r == 255 and 100 < g < 160 and g == b, (r, g, b, a)
assert composite.getpixel((5, 5)) == (255, 255, 255, 255)
print("Example 2 completed")

## Example 3: Filters and enhancements

In [ ]:
from PIL import Image, ImageFilter, ImageEnhance, ImageDraw

# Example 3: Filters and effects
print("Example 3: Filters and effects")

# Create test image
test_img = Image.new('RGB', (200, 150), color='white')
draw = ImageDraw.Draw(test_img)
draw.rectangle([50, 50, 150, 100], fill='red')
draw.ellipse([100, 60, 180, 140], fill='blue')

# Apply filters
filters = [ImageFilter.BLUR, ImageFilter.SHARPEN, ImageFilter.EMBOSS]
for filt in filters:
    result = test_img.filter(filt)
    print(f"Applied {filt.name} filter")
    assert result.size == test_img.size and result.mode == test_img.mode

# Gaussian blur
blurred = test_img.filter(ImageFilter.GaussianBlur(radius=2.0))
print("Applied Gaussian blur")
assert blurred.getpixel((10, 10)) == (255, 255, 255)  # far from any edge: unchanged
assert blurred.getpixel((50, 75)) != test_img.getpixel((50, 75))  # on the red edge: smoothed

# Enhancements
enhancer = ImageEnhance.Brightness(test_img)
bright = enhancer.enhance(1.5)
print("Enhanced brightness")
assert bright.getpixel((10, 10)) == (255, 255, 255)  # already at full brightness: clipped
dark = enhancer.enhance(0.5)
assert dark.getpixel((10, 10)) == (127, 127, 127), dark.getpixel((10, 10))
assert enhancer.enhance(1.0).tobytes() == test_img.tobytes()  # factor 1 is the original

print("Example 3 completed")

## Example 4: Saving and reopening in several formats

In [ ]:
from PIL import Image, ImageDraw
import io

# Example 4: Format support
print("Example 4: Format support")

# Create sample image
img = Image.new('RGB', (100, 100), color='lightgray')
draw = ImageDraw.Draw(img)
draw.rectangle([20, 20, 80, 80], fill='red')

# Test formats
formats = ['JPEG', 'PNG', 'BMP', 'WEBP']
for fmt in formats:
    buffer = io.BytesIO()
    img.save(buffer, format=fmt)
    size = len(buffer.getvalue())
    print(f"{fmt}: {size} bytes")
    # Re-open what was written
    buffer.seek(0)
    reopened = Image.open(buffer)
    reopened.load()
    assert reopened.format == fmt, reopened.format
    assert reopened.size == (100, 100) and reopened.mode == 'RGB', (reopened.size, reopened.mode)
    if fmt in ('PNG', 'BMP'):  # lossless formats keep every pixel
        assert reopened.tobytes() == img.tobytes(), fmt
    else:  # lossy formats stay close to the original colour
        r, g, b = reopened.getpixel((50, 50))
        assert r > 240 and g < 15 and b < 15, (fmt, (r, g, b))

# JPEG quality
jpeg_sizes = {}
for quality in [50, 85, 95]:
    buffer = io.BytesIO()
    img.save(buffer, format='JPEG', quality=quality)
    jpeg_sizes[quality] = len(buffer.getvalue())
    print(f"JPEG q{quality}: {len(buffer.getvalue())} bytes")
assert jpeg_sizes[50] < jpeg_sizes[95], jpeg_sizes  # higher quality, larger file

print("Example 4 completed")

## Example 5: Flips, transposes and blending

In [ ]:
from PIL import Image, ImageDraw
import time

# Example 5: Advanced image operations
print("Example 5: Advanced operations")

# Create test image
base_img = Image.new('RGB', (200, 200), color='lightblue')
draw = ImageDraw.Draw(base_img)
draw.rectangle([50, 50, 150, 150], fill='red')
base_img.putpixel((10, 20), (0, 255, 0))  # an off-centre marker to follow through each transform

# Geometric transformations
transforms = [
    ('Flip H', Image.FLIP_LEFT_RIGHT),
    ('Flip V', Image.FLIP_TOP_BOTTOM),
    ('Transpose', Image.TRANSPOSE),
]

marker_after = {'Flip H': (189, 20), 'Flip V': (10, 179), 'Transpose': (20, 10)}
for name, method in transforms:
    result = base_img.transpose(method)
    print(f"{name}: {result.size}")
    assert result.size == (200, 200)
    assert result.getpixel(marker_after[name]) == (0, 255, 0), name

# Image blending
img1 = Image.new('RGB', (100, 100), color='red')
img2 = Image.new('RGB', (100, 100), color='blue')
blended = Image.blend(img1, img2, 0.5)
print(f"Blended: {blended.size}")
assert blended.getpixel((0, 0)) == (127, 0, 127), blended.getpixel((0, 0))
assert Image.blend(img1, img2, 0.0).getpixel((0, 0)) == (255, 0, 0)
assert Image.blend(img1, img2, 1.0).getpixel((0, 0)) == (0, 0, 255)

print("Example 5 completed")